# Challenge: Promotions

In this challenge, you'll develop codes to parse and analyze data returned from another API on Zalando such as [(Men's Promotions)](https://en.zalando.de/outlet-mens/) or [(Women's Promotions)](https://en.zalando.de/outlet-women/). The workflow is almost the same as in the guided lesson but you'll work with different data.

## Obtaining the link

Wrote your codes in the cell below to obtain the data from the API endpoint you choose. A recap of the workflow:

1. Examine the webpages and choose one that you want to work with.

1. Use Google Chrome's DevTools to inspect the XHR network requests. Find out the API endpoint that serves data to the webpage.

1. Test the API endpoint in the browser to verify its data.

1. Change the page number offset of the API URL to test if it's working.

In [3]:
# your code here - bash first
# https://en.zalando.de/api/catalog/articles?categories=mens-clothing&limit=84&offset=0&sort=popularity



## Reading the data

In the next cell, use Python to obtain data from the API endpoint you chose in the previous step. Workflow:

1. Import libraries.

1. Define the initial API endpoint URL.

1. Make request to obtain data of the 1st page. Flatten the data and store it in an empty object variable.

1. Find out the total page count in the 1st page data.

1. Use a FOR loop to make requests for the additional pages from 2 to page count. Append the data of each additional page to the flatterned data object.

1. Print and review the data you obtained.

In [4]:
# your code here
import requests
import pandas as pd

# 1. Define the initial API endpoint URL (men’s outlet promotions, page 1)
base_url = "https://en.zalando.de/api/catalog/articles"
params = {
    "categories": "mens-clothing",
    "limit": 84,      # number of products per request
    "offset": 0,      # pagination offset
    "sort": "popularity"
}

# 2. Make request to obtain data of the 1st page
response = requests.get(base_url, params=params)
data = response.json()

# Flatten product data
flat_data = pd.json_normalize(data['articles'])   # DataFrame with product details

# 3. Find out the total number of items and pages
total_items = data.get('totalCount', 0)           # total number of products
limit = params["limit"]
total_pages = (total_items // limit) + 1

print(f"Total items: {total_items}")
print(f"Total pages: {total_pages}")

# 4. Loop through additional pages
for page in range(1, total_pages):  # already got page 1, start from 2nd page
    params["offset"] = page * limit
    response = requests.get(base_url, params=params)
    page_data = response.json()
    
    # Append flattened articles to flat_data
    temp_df = pd.json_normalize(page_data['articles'])
    flat_data = pd.concat([flat_data, temp_df], ignore_index=True)

# 5. Print and review the final dataset
print(flat_data.head())
print(flat_data.shape)  # rows = total items, columns = attributes


KeyError: 'articles'

In [5]:
import math
import time
import requests
import pandas as pd
from urllib.parse import urlsplit, urlunsplit, parse_qs, urlencode

# ========= 1) Import libraries — done above =========

# ========= 2) Define the initial API endpoint URL (PASTE YOURS) =========
BASE_URL_WITH_QUERY = "https://en.zalando.de/api/...your-found-endpoint...&limit=84&offset=0"

# Optional: headers help some endpoints behave like a browser
HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-DE,en;q=0.9"
}

# ---- helpers -------------------------------------------------------------
def update_query(url: str, **updates) -> str:
    parts = urlsplit(url)
    q = parse_qs(parts.query, keep_blank_values=True)
    # parse_qs makes values lists; normalize and apply updates
    for k, v in updates.items():
        q[str(k)] = [str(v)]
    new_query = urlencode([(k, v[0]) for k, v in q.items()])
    return urlunsplit((parts.scheme, parts.netloc, parts.path, new_query, parts.fragment))

def get_query_params(url: str):
    q = parse_qs(urlsplit(url).query, keep_blank_values=True)
    # return scalar values where possible
    return {k: (v[0] if isinstance(v, list) and v else v) for k, v in q.items()}

# Recursively find the first "list of dicts" that looks like items
def find_items_list(obj, path=()):
    # Heuristic: product-like keys frequently seen
    productish_keys = {"id","name","brand","brandName","modelId","sku","price","prices","units","media","url","slug"}
    if isinstance(obj, list) and obj and all(isinstance(x, dict) for x in obj):
        # Check if it looks product-ish (not required but improves accuracy)
        sample_keys = set().union(*(x.keys() for x in obj[:5]))
        if sample_keys & productish_keys or len(obj) > 10:
            return obj, path
    if isinstance(obj, dict):
        for k, v in obj.items():
            found = find_items_list(v, path + (k,))
            if found:
                return found
    if isinstance(obj, list):
        for i, v in enumerate(obj):
            found = find_items_list(v, path + (i,))
            if found:
                return found
    return None

# Find a total count int under common keys (recursively)
def find_total_count(obj):
    candidate_keys = {"totalCount","total","total_results","totalElements","totalArticles","count","total_items"}
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in candidate_keys and isinstance(v, int) and v >= 0:
                return v
        for v in obj.values():
            tc = find_total_count(v)
            if tc is not None:
                return tc
    elif isinstance(obj, list):
        for v in obj:
            tc = find_total_count(v)
            if tc is not None:
                return tc
    return None

def request_json(url):
    r = requests.get(url, headers=HEADERS, timeout=25)
    r.raise_for_status()
    try:
        return r.json(), r.headers
    except ValueError:
        raise RuntimeError("Response is not JSON. The endpoint might require different headers or cookies.")

# ========= 3) Request page 1, flatten, store =========
params0 = get_query_params(BASE_URL_WITH_QUERY)
limit = int(params0.get("limit", 84))  # fallback if not present
offset = params0.get("offset")
page = params0.get("page")

data1, headers1 = request_json(BASE_URL_WITH_QUERY)

# Locate the items list dynamically
result = find_items_list(data1)
if not result:
    # print top-level keys to help debug if needed
    raise KeyError(f"Could not locate a list of items. Top-level keys: {list(data1)[:20]}")
items1, path1 = result

flat_data = pd.json_normalize(items1)  # flattened DataFrame for page 1

# ========= 4) Determine total pages =========
# Try: total from JSON, else from headers, else fall back to unknown (iterate until short page)
total_items = find_total_count(data1)
if total_items is None:
    # some APIs include totals in headers (e.g., X-Total-Count)
    for h in ["X-Total-Count", "x-total-count", "X-Count", "x-count"]:
        if h in headers1 and headers1[h].isdigit():
            total_items = int(headers1[h])
            break

if total_items is not None and limit:
    total_pages = int(math.ceil(total_items / float(limit)))
else:
    total_pages = None  # unknown yet; we'll iterate until a short page occurs

print(f"Discovered items path: {path1}")
print(f"Page 1 items: {len(items1)}")
print(f"Total items (if available): {total_items}")
print(f"Computed total pages (if available): {total_pages}")

# ========= 5) Loop through pages 2..N and append =========
all_df = [flat_data]

def build_url_for_page(base_url, page_index_0):
    if offset is not None:
        return update_query(base_url, offset=page_index_0 * limit)
    elif page is not None:
        # many APIs are 1-based pages; we already fetched page 1 when page_index_0==0
        return update_query(base_url, page=page_index_0 + 1)
    else:
        # Default to offset-style if neither present
        return update_query(base_url, offset=page_index_0 * limit)

# Determine how far to go
max_pages_safe_cap = 200  # safety cap to avoid infinite loops
if total_pages is None:
    # keep going until we receive < limit items
    page_idx = 1  # we already fetched index 0
    while page_idx < max_pages_safe_cap:
        url_i = build_url_for_page(BASE_URL_WITH_QUERY, page_idx)
        d_i, _ = request_json(url_i)
        found_i = find_items_list(d_i)
        if not found_i:
            break
        items_i, _ = found_i
        df_i = pd.json_normalize(items_i)
        all_df.append(df_i)
        print(f"Fetched page {page_idx+1}, items: {len(df_i)}")
        if limit and len(df_i) < limit:
            # last page reached
            break
        page_idx += 1
else:
    for page_idx in range(1, min(total_pages, max_pages_safe_cap)):
        url_i = build_url_for_page(BASE_URL_WITH_QUERY, page_idx)
        d_i, _ = request_json(url_i)
        found_i = find_items_list(d_i)
        if not found_i:
            print(f"Warning: no items found on page {page_idx+1}; stopping.")
            break
        items_i, _ = found_i
        df_i = pd.json_normalize(items_i)
        all_df.append(df_i)
        print(f"Fetched page {page_idx+1} / {total_pages}, items: {len(df_i)}")
        # polite pause to be nice to the server
        time.sleep(0.2)

final_df = pd.concat(all_df, ignore_index=True)

# ========= 6) Print & review =========
print(final_df.head(10))
print(f"Final shape: {final_df.shape}")
# Optionally save:
# final_df.to_csv("zalando_mens_promotions.csv", index=False)


HTTPError: 403 Client Error: Forbidden for url: https://en.zalando.de/api/...your-found-endpoint...&limit=84&offset=0

## Bonus

Extract the following information from the data:

* The trending brand.

* The product(s) with the highest discount.

* The sum of discounts of all goods (sum_discounted_prices divided by sum_original_prices).

In [6]:
# your code here
import pandas as pd

# ---- 1. Extract useful fields from the flattened DataFrame ----
# Depending on Zalando's schema, adjust field names:
# - brand name often in "brand.name" or "brand"
# - original price often in "units[0].price.original" or "price.original"
# - discounted price often in "units[0].price.value" or "price.value"

# Try to guess best matching columns
brand_col = [c for c in final_df.columns if "brand" in c.lower()][0]
orig_price_col = [c for c in final_df.columns if "original" in c.lower() and "price" in c.lower()][0]
disc_price_col = [c for c in final_df.columns if "price" in c.lower() and not "original" in c.lower()][0]

df = final_df[[brand_col, orig_price_col, disc_price_col]].copy()
df.columns = ["brand", "original_price", "discounted_price"]

# Convert to numeric
df["original_price"] = pd.to_numeric(df["original_price"], errors="coerce")
df["discounted_price"] = pd.to_numeric(df["discounted_price"], errors="coerce")

# Drop rows without prices
df = df.dropna(subset=["original_price", "discounted_price"])

# ---- 2. Trending brand ----
trending_brand = df["brand"].value_counts().idxmax()

# ---- 3. Highest discount product(s) ----
df["discount_pct"] = (df["original_price"] - df["discounted_price"]) / df["original_price"] * 100
max_discount = df["discount_pct"].max()
top_discount_products = df[df["discount_pct"] == max_discount]

# ---- 4. Overall discount ratio ----
sum_discounted = df["discounted_price"].sum()
sum_original = df["original_price"].sum()
overall_discount_ratio = sum_discounted / sum_original

# ---- 5. Print results ----
print(f"🔥 Trending brand: {trending_brand}")
print(f"📉 Highest discount: {max_discount:.1f}%")
print("Top discounted products:")
print(top_discount_products.head(10)[["brand", "original_price", "discounted_price", "discount_pct"]])

print(f"💰 Overall discount ratio (sum_discounted / sum_original): {overall_discount_ratio:.2%}")


NameError: name 'final_df' is not defined